In [1]:
using Lattices
using HDF5
using JLD2, LinearAlgebra, SparseArrays

include("trotter.jl")
using .Trotter
include("utility_functions.jl")
include("ed_objects.jl")
include("ed_functions.jl")
include("data_path.jl")

data_folder

In [4]:

use_ref = true
antihermitian = true
for folder in readdir(get_data_root())
    dim_parsed = parse_lattice_dimension(folder)
    N_ud = parse_electron_count(folder)

    N_sites = prod(dim_parsed)
    if N_sites >= 9
        continue
    end
    ref = use_ref ? "ref_slater_" : ""
    ah = antihermitian ? "antihermitian_" : ""

    i = 25
    full_folder_name = data_folder(folder)
    local shared_data, target_vecs, indexer

    try
        foldername = joinpath(full_folder_name, "trotter_N=$(N_sites)_$(ref)$(ah)shared.jld2")
        # foldername = joinpath(full_folder_name,"deleteme__shared.jld2")
        _, target_vecs, indexer, _, _, _, _, _ = load_ED_data(full_folder_name; verbose=false, use_slater_reference=use_ref, sign_convention=:spin_first)
        shared_data = load(foldername)["dict"]
        println("folder: $folder")
    catch e
        continue
    end
    iter_data = load(joinpath(full_folder_name, "trotter_N=$(N_sites)_$(ref)$(ah)u_$i.jld2"))["dict"]
    # iter_data = load(joinpath(full_folder_name, "deleteme__u_25.jld2"))["dict"]

    global A_base = iter_data["coefficients"]
    global gates = shared_data["gates"]
    # gates = Trotter.enumerate_ferm_excitations(2, dim_parsed; conserve_mom=true, conserve_sz=true, include_diagonal=!antihermitian)
    global state1 = target_vecs[1, :]
    global state2 = target_vecs[i, :]
    baseline = 1-abs2(state1'*state2)
    println("baseline: |<ref|target>|^2 = $baseline")

    global basis_ints = Trotter.get_basis_sector(indexer, dim_parsed, N_sites)
    loss = Trotter.adjoint_loss(A_base, gates, nothing, state1, state2, basis_ints, N_sites; antihermitian=antihermitian)
    println("Optimized loss: $loss")
    println()
end

folder: N=(2, 2)_2x2
baseline: |<ref|target>|^2 = 0.5012206650433393
Optimized loss: 1.0617978496441083e-8

folder: N=(2, 2)_3x2
baseline: |<ref|target>|^2 = 0.50063543177005
Optimized loss: 2.3762800342996826e-6

folder: N=(3, 3)_3x2
baseline: |<ref|target>|^2 = 0.010929083821033414
Optimized loss: 4.20808055068278e-10

folder: N=(3, 3)_3x2_3
baseline: |<ref|target>|^2 = 0.010929083821033414
Optimized loss: 3.609875176557864e-8

folder: N=(3, 3)_4x2
baseline: |<ref|target>|^2 = 0.004400960850050151
Optimized loss: 3.36631711173041e-7



In [10]:
folder = data_folder("N=(2, 2)_2x2")
u_start = 1
u_end = 25
maxiters = 10
loss_type = :overlap
num_exponentials = 1
antihermitian = true
is_slater_ref = true
# 1. Load ED data (loads indexer if JLD2, or we can use it to build the sector basis)
U_values, state_vecs, indexer, _, N_elec, spin_conserved, _, sign_convention =
    load_ED_data(folder; verbose=true, sign_convention=:spin_first, use_slater_reference=is_slater_ref)

n_up, n_dn = N_elec

# Parse dimension from folder name, default to (3, 3) if fails
Lvec = parse_lattice_dimension(folder)
N_sites = prod(Lvec)

# 2. Computing the basis
# Convert each (up_coords_set, dn_coords_set) entry in the indexer to a combined
# 2N-bit integer that fgateToTauSector expects: up bits in the low N bits, dn bits
# in the upper N bits (via combineSpinInts).
basis_sector = Trotter.get_basis_sector(indexer, Lvec, N_sites)

# 3. Find the Hamiltonian
# Derive the momentum sector from the indexer (same convention as trotter_exp_testing.jl).
# indexer.k is 1-based coordinate tuple; q_target is the C-order flat index (0-based).
q_target = nothing
if !isnothing(indexer.k) && !isnothing(indexer.lattice_dims)
    q_target = Trotter.ravel_c(Tuple(k - 1 for k in indexer.k), Tuple(Lvec))
end
@time H_hop_sector, basis_dict_sector, _ = Trotter.HubbardMomentumBasis(
    1.0, 0.0, Lvec, (n_up, n_dn); q_target=q_target
)
@time H_int_sector, _, _ = Trotter.HubbardMomentumBasis(
    0.0, 1.0, Lvec, (n_up, n_dn); q_target=q_target
)

# 4. Enumerate Trotter gates and tau terms
@time gates = Trotter.enumerate_ferm_excitations(2, Lvec; conserve_mom=true, conserve_sz=true, include_diagonal=!antihermitian)
@time tau_terms = Trotter.fgateToTauSector(gates, N_sites, basis_sector; antihermitian=antihermitian)

scan_instructions = Dict{String,Any}(
    "starting level" => 1,
    "ending level" => 1,
    "num_exponentials" => num_exponentials,
    "antihermitian" => antihermitian,
    "u_range" => 25:25
)

# 6. Run scan optimization
# data_dict = Trotter.interaction_scan_map_to_state(
#     state_vecs, scan_instructions, gates, tau_terms, basis_sector, N_sites;
#     maxiters=10,
#     optimizer=[:LBFGS, :GradientDescent, :LBFGS],
#     initialization_samples=10,
#     H_hopping=H_hop_sector, H_interaction=H_int_sector,
#     save_folder=folder, save_name="deleteme_",
#     loss_type=loss_type,
#     U_values=U_values,
#     antihermitian=antihermitian,
# )


Meta data:
Finding best energy sector
U=1.0e-5 k=[1, 4] [-7.999992500005064, -3.999995000003122]
U=0.001 k=[1, 4] [-7.99925005077831, -3.999500031249997]
U=0.0013433993325989001 k=[1, 4] [-7.998992542139453, -3.9993283567312536]
U=0.001804721766827172 k=[1, 4] [-7.998646624053228, -3.9990977408984785]
U=0.0024244620170823282 k=[1, 4] [-7.998181951938437, -3.998787952679455]
U=0.0032570206556597828 k=[1, 4] [-7.99755777310383, -3.9983718211778907]
U=0.004375479375074184 k=[1, 4] [-7.996719362421174, -3.997812858588034]
U=0.0058780160722749115 k=[1, 4] [-7.9955932418975655, -3.997062071684744]
U=0.007896522868499725 k=[1, 4] [-7.99408077287508, -3.996053687161318]
U=0.010608183551394472 k=[1, 4] [-7.9920495734352315, -3.994699424896451]
U=0.014251026703029992 k=[1, 4] [-7.989322034751655, -3.992880833261015]
U=0.019144819761699575 k=[1, 4] [-7.985659977189414, -3.990439043981614]
U=0.025719138090593455 k=[1, 4] [-7.980744187120196, -3.9871611019657935]
U=0.03455107294592222 k=[1, 4] [-7.

Dict{String, Any} with 8 entries:
  "U_values"       => [1.0e-5, 0.001, 0.0013434, 0.00180472, 0.00242446, 0.0032…
  "maxiters"       => 200
  "optimizer"      => "BFGS"
  "sites"          => "2x2"
  "solver"         => "Lanczos"
  "basis"          => "adiabatic"
  "bc"             => "periodic"
  "electron count" => (2, 2)

Dict{String, Any} with 5 entries:
  "num_exponentials" => 1
  "starting level"   => 1
  "antihermitian"    => true
  "u_range"          => 25:25
  "ending level"     => 1

In [14]:
state_vecs[1,:]'*H_hop_sector*state_vecs[1,:]

0.0 + 0.0im

In [8]:

# using Lattices, SparseArrays, JLD2, LinearAlgebra
# include("trotter.jl")
# using .Trotter
# include("utility_functions.jl")
# using .UtilityFunctions
# include("ed_objects.jl")
# include("ed_functions.jl")
# include("data_path.jl")

folder = data_folder("N=(2, 2)_2x2")
U_values, state_vecs, indexer, _, N_elec, spin_conserved, _, sign_convention = load_ED_data(folder; verbose=false, sign_convention=:spin_first, use_slater_reference=true)
n_up, n_dn = N_elec
Lvec = parse_lattice_dimension("N=(2, 2)_2x2")
N_sites = prod(Lvec)

lattice = Square(Tuple(Lvec), Periodic())
subspace = HubbardSubspace(n_up, n_dn, lattice; k=indexer.k)
H_hop, H_int = create_hubbard_matrices(subspace; indexer=indexer, sign_convention=:spin_first)

state1 = state_vecs[1, :]
state2 = state_vecs[2, :] # U_1 = 0.001

e1 = real(state1' * H_hop * state1)
println("H_hop expectation value on state_vecs[1, :]: ", e1)

H_tot = H_hop + U_values[2] * H_int
e2 = real(state2' * H_tot * state2)
println("H_tot expectation value on state_vecs[2, :]: ", e2)

evals = eigvals(Matrix(H_hop))
println("All eigenvalues of H_hop in sector k=$(indexer.k): ", evals[1:min(5, length(evals))])
println("Minimum eigenvalue of H_hop in sector k=$(indexer.k): ", minimum(evals))


H_hop expectation value on state_vecs[1, :]: -8.0
H_tot expectation value on state_vecs[2, :]: -7.999250001010545
All eigenvalues of H_hop in sector k=(1, 1): [-8.0, -8.0, 0.0, 0.0, 0.0]
Minimum eigenvalue of H_hop in sector k=(1, 1): -8.0


In [51]:
maximum(-real.(diag(H_hop_sector)))

12.0

In [43]:
state_vecs[1,:]'*H_hop_sector *state_vecs[1,:]

4.000000000000002 + 0.0im

In [40]:
d = load(joinpath(folder, "deleteme__shared.jld2"))["dict"]
gates2 = d["gates"]
d = load(joinpath(folder, "deleteme__u_25.jld2"))["dict"]
d["metrics"]

Dict{String, Any} with 2 entries:
  "energy" => [5.25592]
  "loss"   => [2.84295e-10]

In [ ]:
folder = data_folder("N=(3, 3)_3x2")
d = load(joinpath(folder, "deleteme__shared.jld2"))["dict"]
gates2 = d["gates"]
d = load(joinpath(folder, "deleteme__u_25.jld2"))["dict"]
d["coefficients"]


114-element Vector{Float64}:
 -0.0009523801918895046
 -6.747278017295641e-5
  0.00014610350091853093
  0.00040376689578064867
  0.0001368348816817631
 -0.00018752013254557095
 -0.0009426789992356252
  0.00027360342649281684
 -0.0002468424521285368
 -0.00014856286324883987
  ⋮
  0.0071708439451830626
 -0.0030218046198907146
 -0.0008418289050523048
 -0.000882849246424106
 -0.002580380372899112
 -0.0038817775180997284
  0.0014569218828582213
 -0.00016746052826236143
  0.0012500815459323574

In [ ]:
gates

114-element Vector{Float64}:
 -0.0009523801918895046
 -6.747278017295641e-5
  0.00014610350091853093
  0.00040376689578064867
  0.0001368348816817631
 -0.00018752013254557095
 -0.0009426789992356252
  0.00027360342649281684
 -0.0002468424521285368
 -0.00014856286324883987
  ⋮
  0.0071708439451830626
 -0.0030218046198907146
 -0.0008418289050523048
 -0.000882849246424106
 -0.002580380372899112
 -0.0038817775180997284
  0.0014569218828582213
 -0.00016746052826236143
  0.0012500815459323574

In [ ]:
Trotter.adjoint_loss(d["coefficients"],gates, nothing,state_vecs[1,:], state_vecs[25,:], basis_ints, 6;antihermitian=antihermitian)

0.0011935962211307993

In [26]:
Trotter.adjoint_loss(A_base, gates, nothing, state1, state2, basis_ints, N_sites; antihermitian=antihermitian)

0.01092877333615061